# Settings

In [1]:
alpha = 0.05 

In [2]:
def underline(txt: str) -> str:
    """Return *txt* underlined for terminals that support ANSI codes."""
    return f"\033[4m{txt}\033[0m"

def bold(txt: str) -> str:
    """Return *txt* wrapped in the ANSI code for bold (bright) text."""
    return f"\033[1m{txt}\033[0m"

Loading the Study Results into a Dataframe: 

In [3]:
from Study import participants
from HelperFunctions.get_data import get_data

df = get_data(participants=participants, task_ids=[i for i in range(1,18)])
df["id"] = df["id"].apply(lambda x: x.replace("task_", "")).astype(int)

Does not know what IF is, question skipped after trying to figure it out skipped
Skipped, too complicated skipped
Skipped, too complicated skipped
Skipped, doesn't know max skipped
Participant found the -1 confusing, skipped skipped
Did not understand, skipped skipped
Participant was overwhelmed, skipped the question skipped
Participant did not understand, skipped  skipped


-> the data frame includes the columns: 
- "id" for the task id
- "participant" for the participants IDs
- the measured "TTU"
- the measured "DOU"
- information about the tasks content 

# 1. Mixed ANOVA 
- task id = within subjects factor 
- participant id = between subjects factor 


In [4]:
import pingouin as pg

aov_ttu = pg.mixed_anova(
    data=df,
    dv='TTU',         # dependend variable
    within='id',      # within-subjects-Faktor (Tasks)
    between='group',  # between-subjects-Faktor (Groups)
    subject='participant'
)
print("Mixed ANOVA Results for TTU:\n", aov_ttu)

Mixed ANOVA Results for TTU:
         Source             SS  DF1  DF2            MS         F     p_unc  \
0        group   53537.966387    2   18  26768.983193  1.050920  0.370110   
1           id  407535.949580   16  288  25470.996849  1.875047  0.022473   
2  Interaction  420553.747899   32  288  13142.304622  0.967471  0.521606   

   p_GG_corr       np2       eps sphericity       W_spher        p_spher  
0        NaN  0.104560       NaN        NaN           NaN            NaN  
1   0.184401  0.094342  0.068093      False  3.358873e-27  4.349537e-108  
2        NaN  0.097063       NaN        NaN           NaN            NaN  


This means: 

In [5]:
import pandas as pd
import numpy as np

def _eta2_label_and_range(eta2: float) -> tuple:
    """Return a qualitative label and the numeric range that defines it."""
    if eta2 < 0.01:
        return "very small", "< 0.01"
    elif eta2 < 0.06:
        return "small", "0.01 - 0.06"
    elif eta2 < 0.14:
        return "medium", "0.06 - 0.14"
    else:
        return "large", "≥ 0.14"

def _choose_p(row: pd.Series, alpha: float) -> tuple:
    """
    Return the p‑value that should be used for the decision,
    together with a short note why that p‑value was chosen.
    """
    source = row["Source"]
    if source == "id":                     # within‑subject factor → check sphericity
        sphero_violation = row["p_spher"] < alpha
        if sphero_violation:
            return row["p_GG_corr"], f"assumption of sphericity {bold('fails')}: \n\t\tp_spher = {row['p_spher']:.4g} < alpha (Mauchly) \n\t\t-> Green‑Geisser {bold('correction')} (ε = {row['eps']:.3f}) applied", "p_GG_corr"
        else:
            return row["p_unc"], f"assumption of sphericity {bold('holds')}, {bold('no')} correction needed (p_spher >= alpha)", "p_unc"
    else:
        return row["p_unc"], f"{bold('no')} correction needed (factor not within-subject)", "p_unc"




def print_mixed_anova_report_restructured(aov_df: pd.DataFrame,
                                          alpha: float = 0.05,
                                          dv_name: str = "Dependent variable") -> None:
    """
    Print a concise, conclusion‑first report for each factor in a mixed ANOVA.
    All numbers are taken directly from the ANOVA table.
    """
    print(f"\n=== Mixed‑ANOVA summary for {dv_name} (α = {alpha}) ===\n")

    for _, row in aov_df.iterrows():
        source   = row["Source"]
        df1      = int(row["DF1"])
        n_levels = df1 + 1                     # DF1 + 1 = number of levels

        # ----- Choose the appropriate p‑value and note the correction -----
        p_used, corr_note, p_name = _choose_p(row, alpha)

        # ----- Build the logical condition α ? p -----
        if p_used < alpha:
            # significant → α < p
            condition = f"{alpha:.2f} < {p_used:.4f}"
            significance = "significant difference"
        else:
            # not significant → α > p
            condition = f"{alpha:.2f} > {p_used:.4f}"
            significance = "no significant difference"

        # ----- Effect size label and its numeric range -----
        eta2, size_range = _eta2_label_and_range(row["np2"])
        size_lbl = eta2

        print(f"{underline(source)} (levels = DF1 + 1 = {n_levels}):")
        # 1. conclusion + correction note
        print(f"  • {corr_note}: {p_name} = {p_used:.4f} {'≥' if p_used >= alpha else '<'} α")
        # 2. why the conclusion holds (the logical comparison)
        print(f"  • {bold(significance)}: α {'>=' if p_used >= alpha else '<'} "
              f"{p_name} = {p_used:.4f} ({condition})")
        # 3. effect‑size interpretation
        print(f"  • {bold(size_lbl)} explained variance (partial η² = {row['np2']:.3f} in range {size_range})")
        print()   # blank line for readability

In [6]:
print_mixed_anova_report_restructured(aov_ttu, alpha=alpha, dv_name="TTU (Time‑to‑Understand)")


=== Mixed‑ANOVA summary for TTU (Time‑to‑Understand) (α = 0.05) ===

group (levels = DF1 + 1 = 3):
  • no correction needed (factor not within-subject): p_unc = 0.3701 ≥ α
  • no significant difference: α >= p_unc = 0.3701 (0.05 > 0.3701)
  • medium explained variance (partial η² = 0.105 in range 0.06 - 0.14)

id (levels = DF1 + 1 = 17):
  • assumption of sphericity fails: 
		p_spher = 4.35e-108 < alpha (Mauchly) 
		-> Green‑Geisser correction (ε = 0.068) applied: p_GG_corr = 0.1844 ≥ α
  • no significant difference: α >= p_GG_corr = 0.1844 (0.05 > 0.1844)
  • medium explained variance (partial η² = 0.094 in range 0.06 - 0.14)

Interaction (levels = DF1 + 1 = 33):
  • no correction needed (factor not within-subject): p_unc = 0.5216 ≥ α
  • no significant difference: α >= p_unc = 0.5216 (0.05 > 0.5216)
  • medium explained variance (partial η² = 0.097 in range 0.06 - 0.14)



In [7]:
aov_dou = pg.mixed_anova(
    data=df,
    dv='DOU',         # dependend variable
    within='id',      # within-subjects-Faktor (Tasks)
    between='group',  # between-subjects-Faktor (Groups)
    subject='participant'
)
print("Mixed ANOVA Results for DOU:\n", aov_dou)

Mixed ANOVA Results for DOU:
         Source        SS  DF1  DF2        MS         F         p_unc  \
0        group  0.298787    2   18  0.149393  2.308863  1.280596e-01   
1           id  1.910523   16  288  0.119408  4.717518  1.734787e-08   
2  Interaction  1.996134   32  288  0.062379  2.464456  4.507493e-05   

        np2       eps  
0  0.204164       NaN  
1  0.207660  0.289375  
2  0.214965       NaN  


In [8]:
# print_mixed_anova_report_restructured(aov_dou, alpha=alpha, dv_name="DOU (Degree‑of‑Understanding)")
# TODO failes, as there is no p_spherical anymore 

TODO:
find out whether there are differences between groups for certain tasks 
(post hoc and CLD) 

In [15]:
import pandas as pd
import numpy as np
import pingouin as pg
import warnings
warnings.filterwarnings('ignore')   # Pingouin gibt häufig nur Info‑Warnungen aus

# -------------------------------------------------
# 1. Hilfsfunktion: ANOVA für einen einzelnen Task
# -------------------------------------------------
def anova_one_task(task_df, dv='TTU', group_col='group'):
    """
    Führt eine One‑Way‑ANOVA (zwischen den Gruppen) für einen einzelnen Task aus.
    Gibt ein Dictionary zurück, das später zu einem DataFrame zusammengefasst wird.
    """
    # 1) Prüfen, ob mindestens 2 Gruppen im Task vertreten sind
    if task_df[group_col].nunique() < 2:
        # nicht genug Gruppen → kein Test möglich
        return {
            'task_id': task_df['id'].iloc[0],
            'F'      : np.nan,
            'p_unc'  : np.nan,
            'np2'    : np.nan
        }

    # 2) ANOVA ausführen
    aov = pg.anova(dv=dv, between=group_col, data=task_df, detailed=True)

    # 3) Falls das Ergebnis leer ist (z. B. weil nur ein Messwert pro Gruppe),
    #    geben wir NaNs zurück.
    if aov.empty:
        return {
            'task_id': task_df['id'].iloc[0],
            'F'      : np.nan,
            'p_unc'  : np.nan,
            'np2'    : np.nan
        }

    # 4) Wir nehmen die *erste* Zeile (es gibt nur einen Faktor → genau eine Zeile)
    row = aov.iloc[0]

    # 5) Spaltennamen können je nach Pingouin‑Version leicht variieren.
    #    Wir holen sie mit .get() – falls sie nicht existieren, erhalten wir NaN.
    return {
        'task_id': task_df['id'].iloc[0],
        'F'      : row.get('F', np.nan),
        'p_unc'  : row.get('p-unc', row.get('p_unc', np.nan)),   # beide Varianten abdecken
        'np2'    : row.get('np2', np.nan)
    }

# -------------------------------------------------
# 2. Schleife über alle Tasks → Ergebnis‑DataFrame
# -------------------------------------------------
def run_one_way_anova_per_task(df, dv='TTU', group_col='group'):
    """
    Durchläuft alle eindeutigen Task‑IDs, führt für jede Gruppe eine ANOVA aus
    und gibt ein DataFrame mit den rohen p‑Werten zurück.
    """
    results = []
    for task_id, subdf in df.groupby('id'):
        res = anova_one_task(subdf, dv=dv, group_col=group_col)
        results.append(res)
    return pd.DataFrame(results)

# -------------------------------------------------
# 3. Mehrfach‑Test‑Korrektur (Holm‑Bonferroni)
# -------------------------------------------------
def add_multiple_testing_correction(df_results, alpha=0.05, method='holm'):
    """
    Fügt Spalten `p_corr` (korrigierter p‑Wert) und `signif` (True/False) hinzu.
    """
    df = df_results.copy()
    # Nur Zeilen mit einem gültigen p‑Wert berücksichtigen
    valid = df['p_unc'].notna()
    pvals = df.loc[valid, 'p_unc'].values

    # Holm‑Korrektur (oder ein anderes Verfahren, das Pingouin unterstützt)
    p_corr = pg.multicomp(pvals, method=method)

    df.loc[valid, 'p_corr'] = p_corr
    df['signif'] = df['p_corr'] < alpha
    return df

# -------------------------------------------------
# 4. Schöne Ausgabe
# -------------------------------------------------
def pretty_print_task_anova(df_results, dv_name='TTU'):
    print(f"\n=== Ein‑Task‑ANOVA für {dv_name} (α = 0.05) ===\n")
    for _, row in df_results.iterrows():
        task   = int(row['task_id'])
        f_val  = row['F']
        p_raw  = row['p_unc']
        p_corr = row['p_corr']
        eta2   = row['np2']
        signif = row['signif']

        # Effekt‑Größen‑Label
        if pd.isna(eta2):
            size_lbl = "–"
        elif eta2 < 0.01:
            size_lbl = "sehr klein"
        elif eta2 < 0.06:
            size_lbl = "klein"
        elif eta2 < 0.14:
            size_lbl = "mittel"
        else:
            size_lbl = "groß"

        print(f"Task {task:2d}:")
        print(f"  • F = {f_val:.2f}" if not pd.isna(f_val) else "  • F = –")
        print(f"  • p (ungekorrigiert) = {p_raw:.4f}" if not pd.isna(p_raw) else "  • p (ungekorrigiert) = –")
        print(f"  • p (Holm) = {p_corr:.4f}" if not pd.isna(p_corr) else "  • p (Holm) = –")
        print(f"  • Signifikant? {'JA' if signif else 'NEIN'}")
        print(f"  • partial η² = {eta2:.3f} → {size_lbl} Effekt\n")

In [17]:
# -------------------------------------------------
# 2. TTU – Ein‑Task‑ANOVA
# -------------------------------------------------
ttu_task_res = run_one_way_anova_per_task(df, dv='TTU', group_col='group')
print(ttu_task_res)

ttu_task_res = add_multiple_testing_correction(ttu_task_res,
                                                alpha=0.05,
                                                method='holm')
pretty_print_task_anova(ttu_task_res, dv_name='TTU')

# -------------------------------------------------
# 3. DOU – Ein‑Task‑ANOVA (falls gewünscht)
# -------------------------------------------------
dou_task_res = run_one_way_anova_per_task(df, dv='DOU', group_col='group')
dou_task_res = add_multiple_testing_correction(dou_task_res,
                                                alpha=0.05,
                                                method='holm')
pretty_print_task_anova(dou_task_res, dv_name='DOU')

    task_id         F     p_unc       np2
0         1  0.247720  0.783196  0.026787
1         2  3.283636  0.060847  0.267318
2         3  1.497785  0.250209  0.142676
3         4  2.059235  0.156551  0.186200
4         5  1.319220  0.291985  0.127841
5         6  0.968440  0.398600  0.097151
6         7  0.645296  0.536219  0.066903
7         8  0.177739  0.838613  0.019366
8         9  0.062916  0.939228  0.006942
9        10  0.608433  0.555021  0.063323
10       11  0.025279  0.975073  0.002801
11       12  3.100632  0.069648  0.256237
12       13  0.977456  0.395371  0.097966
13       14  1.960055  0.169773  0.178836
14       15  0.007588  0.992444  0.000842
15       16  0.985243  0.392604  0.098670
16       17  0.203390  0.817810  0.022099


ValueError: Must have equal len keys and value when setting with an ndarray